In [1]:
import sys, os
from pathlib import Path

IS_KAGGLE = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '') != ''

if IS_KAGGLE:
    # Install packages not available on Kaggle
    %pip install -q kymatio kornia
    
    # Add repo to path (UPDATE 'deep-learning-course-project' to your dataset slug)
    repo_path = Path('/kaggle/input/deep-learning-course-project')
    if repo_path.exists():
        sys.path.insert(0, str(repo_path))
else:
    # Local: add project root to path (assumes notebook is in notebooks/)
    project_root = Path.cwd().parent
    if (project_root / 'src').exists():
        sys.path.insert(0, str(project_root))

from src.utils.config import *
from src.utils.datasets import get_cifar10_splits, get_cifar10_loaders
from src.models.architectures.RestNet18 import *
from src.models.architectures.ScatNet18 import *
from src.utils.training import *
set_seed(42)


In [2]:
# Hyperparamters
L = 10  # L=10 got best accuracy on full training

DEBUG = True
SKIP_TRAINING = True
EXP_NAME = "finetune_comparison"

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print_progress_every = 1
val_accuracy_storing_threshold = 50

batch_size = 128

# Optimizer
lr = 0.1
momentum = 0.9
weight_decay = 5e-4

# Scheduler
T_max = 200

n_epochs = 200

criterion = nn.CrossEntropyLoss()

In [3]:
class_names = ['Plane', 'Car', 'Bird', 'Cat', 'Deer', 'Dog', 'Frog', 'Horse', 'Ship', 'Truck']
mid_idx = len(class_names) // 2
train_classes = list(range(mid_idx))
finetune_classes = list(range(mid_idx, len(class_names)))

train, val, test = get_cifar10_splits(keep_classes=train_classes)

finetune_train, finetune_val, finetune_test = get_cifar10_splits(keep_classes=finetune_classes)

Files already downloaded and verified
Files already downloaded and verified
Files already downloaded and verified
Files already downloaded and verified


In [4]:
print(len(train), len(val), len(test))
print(len(finetune_train), len(finetune_val), len(finetune_test))

20000 5000 5000
20000 5000 5000


In [5]:
# Set up data loaders
trainloader, valloader, testloader = get_cifar10_loaders(keep_classes=train_classes)

finetune_trainloader, finetune_valloader, finetune_testloader = get_cifar10_loaders(keep_classes=finetune_classes)

# Set up models, optimizers and schedulers
resnet = MakeResNet18(num_classes=len(train_classes)).to(device)
resnet_optimizer, resnet_scheduler = get_optimizer_and_scheduler(resnet)

scat_resnet = MakeScatResNet18(L=L, num_classes=len(train_classes)).to(device)
scatnet_optimizer, scatnet_scheduler = get_optimizer_and_scheduler(scat_resnet)


Files already downloaded and verified
Files already downloaded and verified
Files already downloaded and verified
Files already downloaded and verified


In [ ]:
if not SKIP_TRAINING:
    # resnet_stats = train_model(
    train_model(
        model=resnet,
        trainloader=trainloader,
        valloader=valloader,
        optimizer=resnet_optimizer,
        scheduler=resnet_scheduler,
        criterion=criterion,
        n_epochs=n_epochs,
        device=device,
        experiment_name=EXP_NAME,
        model_name="resnet_first_train",
        DEBUG=DEBUG
    )

    # scatnet_stats = train_model(
    train_model(
        model=scat_resnet,
        trainloader=trainloader,
        valloader=valloader,
        optimizer=scatnet_optimizer,
        scheduler=scatnet_scheduler,
        criterion=criterion,
        n_epochs=n_epochs,
        device=device,
        experiment_name=EXP_NAME,
        model_name="scatnet_first_train",
        DEBUG=DEBUG
    )

In [10]:
# Check models on test
# Load best Performing models
resnet = load_model(resnet, model_name="ResNet_finetune_exp", device=device) #TODO update model name
scat_resnet = load_model(scat_resnet, model_name="ScatNet_finetune_exp", device=device) #TODO update model name